### Import

In [1]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 20
LEVEL = "high"
SEED = 42

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)

print("-"*100); print("[Individual Participation Model optimization]")
x_ind, yp_ind, ym_ind, z_ind, zc_ind, zd_ind, OBJ_IND = optimize_individually_forall(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)

✅ 총 5개 파일을 불러왔습니다: 1201.csv, 137.csv, 401.csv, 524.csv, 89.csv
📊 데이터 Shape: I=5, T=24, S=20
✅ 시뮬레이션 초기화 완료: S=20, Randomness='high', Random Seed=42, M1=763.86, M2=2075.61
----------------------------------------------------------------------------------------------------
[Individual Participation Model optimization]


Optimizing individually for each target_i:   0%|          | 0/5 [00:00<?, ?it/s]

Set parameter Username
Set parameter LicenseID to value 2611964
Academic license - for non-commercial use only - expires 2026-01-20
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i:  20%|██        | 1/5 [00:00<00:00,  9.15it/s]

Optimal solution found for target_i=0! Objective value: 242168.74115753826
Set parameter MIPGap to value 1e-07


...

Optimal solution found for target_i=2! Objective value: 409959.5100997424
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i:  80%|████████  | 4/5 [00:00<00:00,  7.94it/s]

Optimal solution found for target_i=3! Objective value: 463735.6473019376
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i: 100%|██████████| 5/5 [00:00<00:00,  8.23it/s]

Optimal solution found for target_i=4! Objective value: 180377.19278744285


### Linear Decision Rule (Holistic Optimization)

In [25]:
model = gp.Model("holistic_LDR")
model.setParam("MIPGap", 1e-5)
model.setParam(GRB.Param.PoolSearchMode, 2)
model.setParam(GRB.Param.PoolSolutions, 3)

x_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp")
ym_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") 
dm_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z_hol = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc")
zd_hol = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
    
phi1_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") 
phi3_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3") ; phi4_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
phi5_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi5") ; phi6_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")
phi7_hol = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi7")

# LDR 계수 변수들
yp0_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="yp0") ; ypR_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="ypR")
ym0_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="ym0") ; ymR_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="ymR")
dp0_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="dp0") ; dpR_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="dpR")
dm0_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="dm0") ; dmR_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="dmR")
zc0_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zc0") ; zcR_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zcR")
zd0_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zd0") ; zdR_hol = model.addVars(I, T, vtype=GRB.CONTINUOUS, name="zdR")

model.update()

Set parameter MIPGap to value 1e-05
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 3


In [26]:
obj = (gp.quicksum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + 
       gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
model.setObjective(obj, GRB.MAXIMIZE)

# # Quadratic Regularization
# obj = (gp.quicksum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + 
#        gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
# reg = gp.quicksum(x_hol[i, t] * x_hol[i, t] for i in range(I) for t in range(T))
# epsilon = 1e-7
# regularized_obj = obj - epsilon * reg
# model.setObjective(regularized_obj, GRB.MAXIMIZE)

In [27]:
for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(yp_hol[i, t, s] == yp0_hol[i, t] + ypR_hol[i, t] * R[i, t, s])
    model.addConstr(ym_hol[i, t, s] == ym0_hol[i, t] + ymR_hol[i, t] * R[i, t, s])
    model.addConstr(dp_hol[i, t, s] == dp0_hol[i, t] + dpR_hol[i, t] * R[i, t, s])
    model.addConstr(dm_hol[i, t, s] == dm0_hol[i, t] + dmR_hol[i, t] * R[i, t, s])
    model.addConstr(zc_hol[i, t, s] == zc0_hol[i, t] + zcR_hol[i, t] * R[i, t, s])
    model.addConstr(zd_hol[i, t, s] == zd0_hol[i, t] + zdR_hol[i, t] * R[i, t, s])

for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(R[i, t, s] - x_hol[i, t] == yp_hol[i, t, s] - ym_hol[i, t, s] + dp_hol[i, t, s] - dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
    model.addConstr(R[i, t, s] + zd_hol[i, t, s] >= yp_hol[i, t, s] + dp_hol[i, t, s] + zc_hol[i, t, s])
    model.addConstr(zd_hol[i, t, s] <= z_hol[i, t, s])
    model.addConstr(zc_hol[i, t, s] <= K[i] - z_hol[i, t, s])
    model.addConstr(z_hol[i, t, s] <= K[i])
    model.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + 0.92 * zc_hol[i, t, s] - zd_hol[i, t, s] / 0.95)
        
    model.addConstr(yp_hol[i, t, s] <= M1 * phi1_hol[i, t, s]) ; model.addConstr(ym_hol[i, t, s] <= M1 * (1 - phi1_hol[i, t, s]))
    model.addConstr(dp_hol[i, t, s] <= M1 * phi2_hol[i, t, s]) ; model.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi2_hol[i, t, s]))
    model.addConstr(yp_hol[i, t, s] <= M1 * phi3_hol[i, t, s]) ; model.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi3_hol[i, t, s]))
    model.addConstr(ym_hol[i, t, s] <= M1 * phi4_hol[i, t, s]) ; model.addConstr(dp_hol[i, t, s] <= M1 * (1 - phi4_hol[i, t, s]))
    model.addConstr(ym_hol[i, t, s] <= M1 * phi5_hol[i, t, s]) ; model.addConstr(zc_hol[i, t, s] <= M1 * (1 - phi5_hol[i, t, s]))
    model.addConstr(dm_hol[i, t, s] <= M1 * phi6_hol[i, t, s]) ; model.addConstr(zc_hol[i, t, s] <= M1 * (1 - phi6_hol[i, t, s]))
    model.addConstr(zc_hol[i, t, s] <= M1 * phi7_hol[i, t, s]) ; model.addConstr(zd_hol[i, t, s] <= M1 * (1 - phi7_hol[i, t, s]))
    
for i, s in product(range(I), range(S)): model.addConstr(z_hol[i, 0, s] == K0[i])

balance_constraints = {}
for t, s in product(range(T), range(S)):
    balance_constraints[t, s] = model.addConstr(gp.quicksum(dp_hol[i, t, s] for i in range(I)) == gp.quicksum(dm_hol[i, t, s] for i in range(I)), name=f"balance_{t}_{s}")

model.optimize()

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-05
PoolSolutions  3
PoolSearchMode  2

Optimize a model with 62980 rows, 35260 columns and 157780 nonzeros
Model fingerprint: 0x4f25a145
Variable types: 18460 continuous, 16800 integer (16800 binary)
Coefficient statistics:
  Matrix range     [8e-02, 8e+02]
  Objective range  [2e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [8e-02, 8e+02]
Presolve removed 20773 rows and 7045 columns
Presolve time: 0.25s
Presolved: 42207 rows, 28215 columns, 109308 nonzeros
Variable types: 11415 continuous, 16800 integer (16800 binary)
Root relaxation presolved: 42207 rows, 28215 columns, 109308 nonzeros

Deterministic concurrent LP optimizer: primal and dual simplex
Showing primal log only...

Concurrent spin time: 0.16s (can be avoided by choosing Method=3)

Solved with

In [28]:
if model.status == GRB.OPTIMAL:
    num_solutions = model.SolCount
    print(f"\n--- Solution Pool Analysis ---")
    print(f"Found {num_solutions} solutions in the pool.")

    if num_solutions > 1:
        best_obj = model.objVal
        print(f"Best objective value: {best_obj:.8f}\n")

        for i in range(num_solutions):
            model.setParam(GRB.Param.SolutionNumber, i)
            pool_obj = model.PoolObjVal
            diff = best_obj - pool_obj
            
            print(f"Solution {i}: Objective = {pool_obj:.8f},  Difference from best = {diff:.8f}")

    model.setParam(GRB.Param.SolutionNumber, 0)
    
    x_sol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
    yp_sol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_sol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_sol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_sol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_sol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_sol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_sol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = model.objVal
    phi1_sol = np.array([[[phi1_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi2_sol = np.array([[[phi2_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    phi3_sol = np.array([[[phi3_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi4_sol = np.array([[[phi4_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    phi5_sol = np.array([[[phi5_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi6_sol = np.array([[[phi6_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    OBJ_HOL = model.objVal
    
    yp0_sol = np.array([[yp0_hol[i, t].X for t in range(T)] for i in range(I)])
    ypR_sol = np.array([[ypR_hol[i, t].X for t in range(T)] for i in range(I)])
    
    ym0_sol = np.array([[ym0_hol[i, t].X for t in range(T)] for i in range(I)])
    ymR_sol = np.array([[ymR_hol[i, t].X for t in range(T)] for i in range(I)])
    
    dp0_sol = np.array([[dp0_hol[i, t].X for t in range(T)] for i in range(I)])
    dpR_sol = np.array([[dpR_hol[i, t].X for t in range(T)] for i in range(I)])
    
    dm0_sol = np.array([[dm0_hol[i, t].X for t in range(T)] for i in range(I)])
    dmR_sol = np.array([[dmR_hol[i, t].X for t in range(T)] for i in range(I)])
    
    zc0_sol = np.array([[zc0_hol[i, t].X for t in range(T)] for i in range(I)])
    zcR_sol = np.array([[zcR_hol[i, t].X for t in range(T)] for i in range(I)])
    
    zd0_sol = np.array([[zd0_hol[i, t].X for t in range(T)] for i in range(I)])
    zdR_sol = np.array([[zdR_hol[i, t].X for t in range(T)] for i in range(I)])


--- Solution Pool Analysis ---
Found 3 solutions in the pool.
Best objective value: 1560123.67500198

Solution 0: Objective = 1560123.67500198,  Difference from best = 0.00000000
Solution 1: Objective = 1560123.67500198,  Difference from best = 0.00000000
Solution 2: Objective = 1560123.67500198,  Difference from best = 0.00000000


In [29]:
if model.status == GRB.OPTIMAL:        
    try:
        lambda_dual = {}
        for t, s in product(range(T), range(S)): 
            lambda_dual[t, s] = balance_constraints[t, s].Pi
        print("Direct dual extraction successful!")
        
    except AttributeError:
        print("\nDirect dual extraction failed. Using Model.fixed() method...")
        
        fixed_model = model.fixed()
        
        fixed_x_hol = {(i, t): fixed_model.getVarByName(f"x[{i},{t}]") for i, t in product(range(I), range(T))}
        fixed_yp_hol = {(i, t, s): fixed_model.getVarByName(f"yp[{i},{t},{s}]") for i, t, s in product(range(I), range(T), range(S))}
        fixed_ym_hol = {(i, t, s): fixed_model.getVarByName(f"ym[{i},{t},{s}]") for i, t, s in product(range(I), range(T), range(S))}

        linear_obj_for_fixed_model = (
            gp.quicksum(P_DA[t] * fixed_x_hol[i, t] for i, t in product(range(I), range(T))) +
            gp.quicksum((1/S) * (P_RT[t, s] * fixed_yp_hol[i, t, s] - P_PN[t, s] * fixed_ym_hol[i, t, s]) 
                         for i, t, s in product(range(I), range(T), range(S)))
        )
        
        # reg_fixed = gp.quicksum(fixed_x_hol[i, t] * fixed_x_hol[i, t] for i, t in product(range(I), range(T)))
        # regularized_obj_fixed = linear_obj_for_fixed_model - epsilon * reg_fixed
        regularized_obj_fixed = linear_obj_for_fixed_model
        
        fixed_model.setParam("OutputFlag", 0)
        fixed_model.setParam("MIPGap", 1e-5)
        fixed_model.setParam(GRB.Param.PoolSearchMode, 2)
        fixed_model.setParam(GRB.Param.PoolSolutions, 4)
        fixed_model.setObjective(regularized_obj_fixed, GRB.MAXIMIZE)
        fixed_model.optimize()
        
        if fixed_model.status == GRB.OPTIMAL:
            original_obj_val = (
                sum(P_DA[t] * x_sol[i, t] for i, t in product(range(I), range(T))) +
                sum((1/S) * (P_RT[t, s] * yp_sol[i, t, s] - P_PN[t, s] * ym_sol[i, t, s]) 
                    for i, t, s in product(range(I), range(T), range(S)))
                # - epsilon * sum(x_sol[i, t] * x_sol[i, t] for i, t in product(range(I), range(T)))
            )
            
            fixed_obj = fixed_model.objVal
            obj_diff = abs(original_obj_val - fixed_obj)

            print(f"Original Objective: {original_obj_val:.6f}")
            print(f"Fixed Model Objective: {fixed_obj:.6f}")
            print(f"Difference: {obj_diff:.10f}")
            
            if obj_diff < 1e-6:
                print("✅ Objective values match! Fixed model is consistent.")
            else:
                print("⚠️ Warning: Objective values don't match.")

            num_solutions = fixed_model.SolCount
            print(f"\n--- Fixed Model Solution Pool Analysis ---")
            print(f"Found {num_solutions} solutions in the pool.")

            if num_solutions > 1:
                best_obj = fixed_model.objVal
                print(f"Best objective value: {best_obj:.8f}\n")
                for i in range(num_solutions):
                    fixed_model.setParam(GRB.Param.SolutionNumber, i)
                    pool_obj = fixed_model.PoolObjVal
                    diff = abs(best_obj - pool_obj)
                    print(f"Solution {i}: Objective = {pool_obj:.8f},  Difference = {diff:.8f}")
                fixed_model.setParam(GRB.Param.SolutionNumber, 0)

            print("\n=== Solution Comparison ===")
            fixed_vars = {var.VarName: var.X for var in fixed_model.getVars()}
            
            print("=== x values by individual ===")
            print("Individual | Time | Original x | Fixed x | Difference")
            print("-" * 55)
            
            max_x_diff = 0
            for i in range(I):
                for t in range(T):
                    original_x = x_sol[i, t]
                    fixed_x = fixed_vars.get(f"x[{i},{t}]", 0)
                    diff = abs(original_x - fixed_x)
                    max_x_diff = max(max_x_diff, diff)
                    if original_x != 0:
                        print(f"{i:10d} | {t:4d} | {original_x:10.6f} | {fixed_x:7.6f} | {diff:10.8f}")
            
            print("\n=== yp values sum over i (scenario average) ===")
            print("Time | Original yp_sum | Fixed yp_sum | Difference")
            print("-" * 52)
            max_yp_diff = 0
            for t in range(14,16):
                original_yp_sum = sum(sum(yp_sol[i, t, s] for i in range(I)) for s in range(S)) / S
                fixed_yp_sum = sum(sum(fixed_vars.get(f"yp[{i},{t},{s}]", 0) for i in range(I)) for s in range(S)) / S
                diff = abs(original_yp_sum - fixed_yp_sum)
                max_yp_diff = max(max_yp_diff, diff)
                print(f"{t:4d} | {original_yp_sum:14.6f} | {fixed_yp_sum:12.6f} | {diff:10.8f}")

        lambda_dual = np.zeros((T, S))
        if fixed_model.status == GRB.OPTIMAL:
            for t, s in product(range(T), range(S)):
                constr_name = f"balance_{t}_{s}"
                try:
                    constr = fixed_model.getConstrByName(constr_name)
                    if constr is not None:
                        lambda_dual[t, s] = constr.Pi
                    else:
                        lambda_dual[t, s] = np.nan
                except:
                    lambda_dual[t, s] = np.nan 
                    
            print("\nModel.fixed() dual extraction successful!")
        else:
            print("Fixed model optimization failed. Setting dual to zeros.")
            lambda_dual = np.zeros((T, S))

        print("\nInternal Settlement Prices (Dual Variables):")
        for t, s in product(range(T), range(1,2)):
            print(f"λ_{t}(ξ_{s}) = {lambda_dual[t, s]:.4f}")
            
else:
    print("No optimal solution found.")
    lambda_dual = {(t, s): np.nan for t in range(T) for s in range(S)}
    x_sol = yp_sol = ym_sol = dp_sol = dm_sol = z_sol = zc_sol = zd_sol = None
    original_objval = None


Direct dual extraction failed. Using Model.fixed() method...
Original Objective: 1560123.675002
Fixed Model Objective: 1560123.675002
Difference: 0.0000000005
✅ Objective values match! Fixed model is consistent.

--- Fixed Model Solution Pool Analysis ---
Found 1 solutions in the pool.

=== Solution Comparison ===
=== x values by individual ===
Individual | Time | Original x | Fixed x | Difference
-------------------------------------------------------
         0 |   21 |  31.370493 | 31.370493 | 0.00000000
         1 |   20 |  29.182099 | 29.182099 | 0.00000000
         2 |   22 |   0.404690 | 0.404690 | 0.00000000
         3 |   22 |   7.474691 | 7.474691 | 0.00000000

=== yp values sum over i (scenario average) ===
Time | Original yp_sum | Fixed yp_sum | Difference
----------------------------------------------------
  14 |    1213.867952 |  1213.867952 | 0.00000000
  15 |     978.903996 |   978.903996 | 0.00000000

Model.fixed() dual extraction successful!

Internal Settlement Pri

In [30]:
print("\n=== 모든 LDR 계수 (i, t별) ===")
print("Individual | Time | Variable | 상수항     | R계수    ")
print("-" * 50)

for t in range(T):
    for i in range(I):
        print(f"{i:10d} | {t:4d} | yp       | {yp0_sol[i,t]:8.4f} | {ypR_sol[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | ym       | {ym0_sol[i,t]:8.4f} | {ymR_sol[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | dp       | {dp0_sol[i,t]:8.4f} | {dpR_sol[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | dm       | {dm0_sol[i,t]:8.4f} | {dmR_sol[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | zc       | {zc0_sol[i,t]:8.4f} | {zcR_sol[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | zd       | {zd0_sol[i,t]:8.4f} | {zdR_sol[i,t]:7.4f}")
        print(f"{i:10d} | {t:4d} | x (1st)  | {x_sol[i,t]:8.4f} |    -    ")
        print("-" * 50)


=== 모든 LDR 계수 (i, t별) ===
Individual | Time | Variable | 상수항     | R계수    
--------------------------------------------------
         0 |    0 | yp       |   0.0000 |  0.0000
         0 |    0 | ym       |   0.0000 |  0.0000
         0 |    0 | dp       |   0.0000 |  0.0000
         0 |    0 | dm       |   0.0000 |  0.0000
         0 |    0 | zc       |   0.0000 |  0.0000
         0 |    0 | zd       |   0.0000 |  0.0000
         0 |    0 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         1 |    0 | yp       |   0.0000 |  0.0000
         1 |    0 | ym       |   0.0000 |  0.0000
         1 |    0 | dp       |   0.0000 |  0.0000
         1 |    0 | dm       |   0.0000 |  0.0000
         1 |    0 | zc       |   0.0000 |  0.0000
         1 |    0 | zd       |   0.0000 |  0.0000
         1 |    0 | x (1st)  |   0.0000 |    -    
--------------------------------------------------
         2 |    0 | yp       |   0.0000 |  0.0000
         2 |    0 |

In [31]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 90)
print("\n[HOLISTIC]") ; print(header)
for t in range(7, 22):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_sol[:, t].sum()
    yp_avg = np.mean([yp_sol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_sol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_sol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_sol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_sol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_sol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_sol[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[HOLISTIC]
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 7 |    12.59     0.00     0.00     0.00     0.00     0.00    12.59     0.00     3.26
 8 |    41.70     0.00     0.86     0.00     0.00     0.00    40.84     0.00    14.85
 9 |   185.31     0.00   113.93     0.00     0.00     0.00    71.38     0.00    52.42
10 |   445.25     0.00   318.63     0.00     0.00     0.00   126.62     0.00   118.09
11 |   652.06     0.00   562.22     0.00     0.00     0.00    89.84     0.00   234.59
12 |   860.69     0.00   850.31     0.00     0.00     0.00    21.00    10.62   317.24
13 |  1397.66     0.00  1528.19     0.00     0.00     0.00     0.00   130.53   325.38
14 |  1338.06     0.00  1213.87     0.00     0.00     0.00   124.19     0.00   187.97
15 |   804.05     0.00   978.90     0.00     0.00     0.00     0.00   174.86   302.23
16 |   756.56     0.00   711.99     0

In [17]:
data = []
for t, s in product(range(9,17), range(0,1)):
    data.append({
        'Time': t,
        'Scenario': s,
        'P_DA': round(P_DA[t], 2),
        'P_RT': round(P_RT[t, s], 2),
        'Lambda': round(-lambda_dual[t, s] * S, 2),
        'P_PN': round(P_PN[t, s], 2)
    })
pd.DataFrame(data)

,Time,Scenario,P_DA,P_RT,Lambda,P_PN
0,9,0,132.150,88.630,-0.000,264.290
1,10,0,136.680,126.700,-0.000,273.360
2,11,0,142.310,151.010,-0.000,302.020
3,12,0,157.020,230.810,-0.000,461.610
4,13,0,155.480,281.740,-0.000,563.480
5,14,0,170.140,158.050,-0.000,340.280
6,15,0,183.990,208.980,-0.000,417.970
7,16,0,191.700,154.260,-0.000,383.400


In [79]:
lambda_rep = np.zeros((T, S))

data = []
for t in range(T):
    for s in range(S):
        lambda_rep[t, s] = lambda_dual[t, s]
    p_rt_avg = np.mean(P_RT[t, :]) ; p_pn_avg = np.mean(P_PN[t, :])
    
    # lambda_rep[t, :] = np.mean(lambda_rep[t, :])
    # lambda_rep[t, :] = np.mean(lambda_rep[t, :][lambda_rep[t, :] < 0]) if np.any(lambda_rep[t, :] < 0) else 0
    lambda_rep[t, :] = np.mean(lambda_rep[t, :][np.abs(lambda_rep[t, :]) > 0]) if np.any(np.abs(lambda_rep[t, :]) > 0) else 0
    
    data.append({
        'Time': t, 'P_RT_avg': round(p_rt_avg, 2), 'Lambda': round(-lambda_rep[t, 0] * S, 2), 'P_PN_avg': round(p_pn_avg, 2)
    })
pd.DataFrame(data).head()

,Time,P_RT_avg,Lambda,P_PN_avg
0,0,62.570,-0.000,213.440
1,1,87.530,-0.000,196.750
2,2,74.840,-0.000,178.380
3,3,66.830,-0.000,167.120
4,4,72.650,-0.000,168.650


### Individual Replay

In [58]:
model = gp.Model("DER_Individual_Replay")
model.setParam("MIPGap", 1e-4)
model.setParam(GRB.Param.PoolSearchMode, 2)
model.setParam(GRB.Param.PoolSolutions, 4)

x = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym") 
dp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm") 
z = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
phi1 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") 
phi3 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi3") ; phi4 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
phi5 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi5") ; phi6 = model.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")

model.update()

obj = (
    gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) + 
    gp.quicksum((1/S) * (
        P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
    ) for i in range(I) for t in range(T) for s in range(S)) +
    gp.quicksum(lambda_dual[t, s] * (
        gp.quicksum(dm[i, t, s] for i in range(I)) - gp.quicksum(dp[i, t, s] for i in range(I))
    ) for t in range(T) for s in range(S))
)
model.setObjective(obj, GRB.MAXIMIZE)

# epsilon = 1e-7
# reg = gp.quicksum(x[i, t] * x[i, t] for i in range(I) for t in range(T))
# regularized_obj = obj - epsilon * reg
# model.setObjective(regularized_obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    model.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
    model.addConstr(R[i, t, s] + zd[i, t, s] >= yp[i, t, s] + dp[i, t, s] + zc[i, t, s])
    model.addConstr(zd[i, t, s] <= z[i, t, s]) ; model.addConstr(zc[i, t, s] <= K[i] - z[i, t, s]) ; model.addConstr(z[i, t, s] <= K[i])
    model.addConstr(z[i, t + 1, s] == z[i, t, s] + 0.92 * zc[i, t, s] - zd[i, t, s] / 0.95)
    
    # model.addConstr(yp[i, t, s] <= M1 * phi1_sol[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1_sol[i, t, s]))
    # model.addConstr(dp[i, t, s] <= M1 * phi2_sol[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi2_sol[i, t, s]))
    # model.addConstr(yp[i, t, s] <= M1 * phi3_sol[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi3_sol[i, t, s]))
    # model.addConstr(ym[i, t, s] <= M1 * phi4_sol[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi4_sol[i, t, s]))
    # model.addConstr(ym[i, t, s] <= M1 * phi5_sol[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi5_sol[i, t, s]))
    # model.addConstr(dm[i, t, s] <= M1 * phi6_sol[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi6_sol[i, t, s]))
    
    model.addConstr(yp[i, t, s] <= M1 * phi1[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1[i, t, s]))
    model.addConstr(dp[i, t, s] <= M1 * phi2[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi2[i, t, s]))
    model.addConstr(yp[i, t, s] <= M1 * phi3[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi3[i, t, s]))
    model.addConstr(ym[i, t, s] <= M1 * phi4[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi4[i, t, s]))
    model.addConstr(ym[i, t, s] <= M1 * phi5[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi5[i, t, s]))
    model.addConstr(dm[i, t, s] <= M1 * phi6[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi6[i, t, s]))

for i, s in product(range(I), range(S)): model.addConstr(z[i, 0, s] == K0[i])

model.optimize()

if model.status == GRB.OPTIMAL:
    num_solutions = model.SolCount
    print(f"\n--- Solution Pool Analysis ---")
    print(f"Found {num_solutions} solutions in the pool.")

    if num_solutions > 1:
        best_obj = model.objVal
        print(f"Best objective value: {best_obj:.8f}\n")

        for i in range(num_solutions):
            model.setParam(GRB.Param.SolutionNumber, i)
            pool_obj = model.PoolObjVal
            diff = best_obj - pool_obj
            
            print(f"Solution {i}: Objective = {pool_obj:.8f},  Difference from best = {diff:.8f}")

    model.setParam(GRB.Param.SolutionNumber, 0)
    
    print(f"Optimal solution found! Objective value: {model.objVal}")
else:
    print("No optimal solution found.")

x_re = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
yp_re = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_re = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
dp_re = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_re = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
z_re = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
zc_re = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_re = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
OBJ_RE = model.objVal

Set parameter MIPGap to value 0.0001
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 4
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
PoolSolutions  4
PoolSearchMode  2

Optimize a model with 43300 rows, 31420 columns and 105700 nonzeros
Model fingerprint: 0xc6650d22
Variable types: 17020 continuous, 14400 integer (14400 binary)
Coefficient statistics:
  Matrix range     [9e-01, 8e+02]
  Objective range  [2e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [8e-02, 8e+02]
Presolve removed 9540 rows and 3570 columns
Presolve time: 0.05s
Presolved: 33760 rows, 27850 columns, 83710 nonzeros
Variable types: 13450 continuous, 14400 integer (14400 binary)
Found heuristic solution: objective 3939025.0426

Root relaxation: objective 1.386341e+07, 17134 iterations, 0.33 seconds (0.49 work units)


In [59]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 90)
print("\n[REPLAY]") ; print(header)
for t in range(9, 20):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_re[:, t].sum()
    yp_avg = np.mean([yp_re[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_re[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_re[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_re[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")

print("\n[HOLISTIC]") ; print(header)
for t in range(9, 20):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_sol[:, t].sum()
    yp_avg = np.mean([yp_sol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_sol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_sol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_sol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_sol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_sol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_sol[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[REPLAY]
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 9 |   185.31  3995.78     0.00    26.67     0.00  3783.79     0.00     0.00     0.00
10 |   445.25  4271.83     0.00    86.98     0.00  3739.61     0.00     0.00     0.00
11 |   652.06  4481.90     0.00   133.80     0.00  3696.04     0.00     0.00     0.00
12 |   860.69  4548.96     0.00   114.32     0.00  3573.95     0.00     0.00     0.00
13 |  1397.66  4739.88     0.00    65.93     0.00  3276.29     0.00     0.00     0.00
14 |  1338.06  5194.24     0.00   297.28     0.00  3558.90     0.00     0.00     0.00
15 |   804.05  4372.93     0.00    66.93     0.00  3501.95     0.00     0.00     0.00
16 |   756.56  4532.14     0.00   115.97     0.00  3659.61     0.00     0.00     0.00
17 |   738.45  4569.62     0.00   147.42     0.00  3683.75     0.00     0.00     0.00
18 |   493.98  4235.53     0.00    75.7

In [45]:
print("="*50) ; print("AGGREGATOR LOSS ANALYSIS") ; print("="*50)

total_losses = []

for t in range(T):
    scenario_losses = []
    
    for s in range(S):
        total_supply = np.sum(dp_re[:, t, s])
        total_demand = np.sum(dm_re[:, t, s])
        
        if total_supply > total_demand:
            excess = total_supply - total_demand
            # loss = excess * (-lambda_rep[t, s]*S - P_RT[t, s])
            loss = excess * (P_RT[t, s])
        else:
            excess = total_demand - total_supply
            # loss = excess * (P_PN[t, s] - (-lambda_rep[t,s]*S))
            loss = excess * (P_PN[t, s])
        
        scenario_losses.append(loss)
    
    avg_loss = np.mean(scenario_losses)
    total_losses.append(avg_loss)
    # print(f"t={t} Average Loss: {avg_loss:.2f}")

overall_avg_loss = np.mean(total_losses)
total_loss = np.sum(total_losses)

print("[SUMMARY]")
print("Individual Participation Profit", sum(OBJ_IND[i] for i in range(I)))
print("Expected Replay Profit", OBJ_RE)
print(f"Total loss across all time periods: {total_loss:.2f}")
print("Realized Profit", OBJ_RE - total_loss)
print("Holistic Profit", OBJ_HOL)

AGGREGATOR LOSS ANALYSIS
[SUMMARY]
Individual Participation Profit 1654025.1983541553
Expected Replay Profit 1755976.0956941566
Total loss across all time periods: 124517.72
Realized Profit 1631458.3787724331


NameError: name 'OBJ_HOL' is not defined